## Step 9 - OCR + Captioning with Qwen2.5-VL (one JSON per video)

Replaces EasyOCR (step 6) and adds a **caption channel** in ONE inference per
keyframe: **Qwen2.5-VL-3B-Instruct** reads each frame and returns

```json
{"ocr": "<all visible text, verbatim, correct Vietnamese diacritics>",
 "caption": "<1-2 English sentences describing the scene>"}
```

Why Qwen2.5-VL (Vortex, arXiv:2606.19682): far better Vietnamese OCR than
EasyOCR on broadcast frames, and the caption gives a retrieval channel that
survives where visual embeddings fail (unusual objects, abstract queries).

**Outputs** (per-video, resumable like steps 6-8):

```
ocr_caption/<video_name>.json   # { "<frame_id>": {"ocr": "...", "caption": "..."} }
ocr_data.json                   # merged, SAME format as step 6 -> backend data/ocr/
caption_data.json               # merged  { "<frame_id>": "<caption>" } -> backend data/captions/
```

**Cost**: ~1-1.5 s/frame on a T4 -> 55k frames ≈ 4-6 phiên 12h. Resumable per
video AND per frame - just re-run to continue. To quarter the cost set
`FRAME_STRIDE = 4` (caption/OCR mỗi 4 keyframe một lần; keyframe vốn dày nên
mỗi cảnh vẫn được phủ).

Set **Accelerator = GPU T4 x2** (model chạy trên GPU 0).

In [ ]:
# Qwen2.5-VL cần transformers mới + qwen-vl-utils (xử lý ảnh/video message)
!pip install -q -U "transformers>=4.49" qwen-vl-utils accelerate

In [ ]:
import json
import os
import re
from pathlib import Path

import torch
from PIL import Image
from tqdm.auto import tqdm
from transformers import AutoProcessor, Qwen2_5_VLForConditionalGeneration

print('torch', torch.__version__, '| GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'none')

In [ ]:
# Step 3 keyframes (nested: Keyframes/<batch>/<video_name>/<idx>.webp)
KEYFRAMES_DIR    = '/kaggle/input/aic-keyframes-webp/Keyframes'
OUTPUT_DIR       = './ocr_caption'       # one <video_name>.json per video
MODEL_ID         = 'Qwen/Qwen2.5-VL-3B-Instruct'
FRAME_STRIDE     = 1                     # 1 = mọi keyframe; 4 = 1/4 chi phí
MAX_NEW_TOKENS   = 220
IMAGE_EXTS       = ('.jpg', '.jpeg', '.png', '.webp')
CHECKPOINT_EVERY = 50                    # save the video's JSON every N frames

PROMPT = (
    "You are analyzing one keyframe from a Vietnamese TV broadcast.\n"
    "Return ONLY a JSON object with exactly these keys:\n"
    '{"ocr": "<ALL text visible in the image, verbatim, keep Vietnamese diacritics; '
    '\\"\\" if none>",\n'
    ' "caption": "<1-2 English sentences describing the visual scene: subjects, '
    'actions, setting. Do not mention the channel logo or timestamps.>"}\n'
    "No markdown, no explanations, JSON only."
)

os.makedirs(OUTPUT_DIR, exist_ok=True)

In [ ]:
# Load model once (fp16, ~7GB -> vừa T4 16GB)
model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
    MODEL_ID, torch_dtype=torch.float16, device_map='cuda:0'
)
processor = AutoProcessor.from_pretrained(MODEL_ID)
model.eval()
print('model ready:', MODEL_ID)

In [ ]:
def _parse_json(text):
    """Chịu được model lỡ bọc ```json ...``` hoặc thêm chữ quanh JSON."""
    m = re.search(r'\{.*\}', text, flags=re.DOTALL)
    if not m:
        return {}
    try:
        d = json.loads(m.group(0))
    except ValueError:
        return {}
    return {
        'ocr': str(d.get('ocr', '')).strip(),
        'caption': str(d.get('caption', '')).strip(),
    }


@torch.inference_mode()
def ocr_caption_image(path):
    """{'ocr': ..., 'caption': ...} cho một keyframe (một lần inference)."""
    image = Image.open(path).convert('RGB')
    messages = [{'role': 'user', 'content': [
        {'type': 'image', 'image': image},
        {'type': 'text', 'text': PROMPT},
    ]}]
    text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = processor(text=[text], images=[image], return_tensors='pt').to(model.device)
    out = model.generate(**inputs, max_new_tokens=MAX_NEW_TOKENS, do_sample=False)
    reply = processor.batch_decode(
        out[:, inputs.input_ids.shape[1]:], skip_special_tokens=True
    )[0]
    return _parse_json(reply)

# smoke test
_first = next(p for p in sorted(Path(KEYFRAMES_DIR).rglob('*')) if p.suffix in IMAGE_EXTS)
print(_first.name, '->', ocr_caption_image(_first))

In [ ]:
# Resume 2 cấp như step 6/8: video đủ frame thì bỏ qua; trong video chỉ chạy
# frame còn thiếu. FRAME_STRIDE áp trên danh sách frame đã sort theo idx.
video_dirs = sorted(d for d in Path(KEYFRAMES_DIR).glob('*/*') if d.is_dir())
print(len(video_dirs), 'videos | stride =', FRAME_STRIDE)

for video_dir in tqdm(video_dirs, desc='videos'):
    video_name = video_dir.name
    out_path = Path(OUTPUT_DIR) / f'{video_name}.json'
    done = json.loads(out_path.read_text(encoding='utf-8')) if out_path.exists() else {}

    frames = sorted(p for p in video_dir.iterdir() if p.suffix in IMAGE_EXTS)
    frames = frames[::FRAME_STRIDE]
    todo = [(f'{video_name}_{p.stem}', p) for p in frames if f'{video_name}_{p.stem}' not in done]
    if not todo:
        continue

    for i, (frame_id, p) in enumerate(tqdm(todo, desc=video_name, leave=False), 1):
        try:
            done[frame_id] = ocr_caption_image(p)
        except Exception as e:                      # một ảnh hỏng không dừng cả run
            print(f'[WARN] {frame_id}: {type(e).__name__}: {e}')
            done[frame_id] = {'ocr': '', 'caption': ''}
        if i % CHECKPOINT_EVERY == 0:
            out_path.write_text(json.dumps(done, ensure_ascii=False), encoding='utf-8')
    out_path.write_text(json.dumps(done, ensure_ascii=False), encoding='utf-8')

print('done:', sum(1 for _ in Path(OUTPUT_DIR).glob('*.json')), 'video JSONs')

In [ ]:
# Merge: ocr_data.json (format step 6, thay thẳng file cũ) + caption_data.json
ocr_merged, cap_merged = {}, {}
for f in sorted(Path(OUTPUT_DIR).glob('*.json')):
    for frame_id, d in json.loads(f.read_text(encoding='utf-8')).items():
        if d.get('ocr'):
            ocr_merged[frame_id] = d['ocr']
        if d.get('caption'):
            cap_merged[frame_id] = d['caption']

with open('ocr_data.json', 'w', encoding='utf-8') as fh:
    json.dump(ocr_merged, fh, ensure_ascii=False)
with open('caption_data.json', 'w', encoding='utf-8') as fh:
    json.dump(cap_merged, fh, ensure_ascii=False)

print(f'ocr_data.json: {len(ocr_merged)} frames | caption_data.json: {len(cap_merged)} frames')

## Use with the backend

```text
ocr_data.json     -> backend/data/ocr/ocr_data.json        (THAY file EasyOCR cũ)
caption_data.json -> backend/data/captions/caption_data.json
```

Rồi build lại các index text + graph (OCR đổi nên scene text trong graph cũng đổi):

```bash
cd data_processing
python 07_build_text_indexes.py    # ocr_text + asr_text + caption_text index
python 08_build_graph.py           # scene.ocr lấy từ OCR mới
```

Backend tự nhận: `/search_ocr` dùng OCR sạch hơn, `/search_caption` + kênh
`caption` trong agent fan-out hoạt động khi `caption_text.index` tồn tại.